<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Llama 3와 Ollama를 통한 지시 데이터셋 생성하기

- 이 노트북은 ollama를 통해 80억 파라미터의 Llama 3 모델을 사용하여 "Magpie: Alignment Data Synthesis from Scratch by Prompting Aligned LLMs with Nothing" 논문([https://arxiv.org/abs/2406.08464](https://arxiv.org/abs/2406.08464))에서 제안된 "핫"을 사용해 합성 데이터셋을 생성합니다

- 생성된 데이터셋은 Alpaca에서 찾을 수 있는 것과 유사한 "instruction"과 "output" 필드가 있는 지시 데이터셋이 될 것입니다:


```python
{
    "instruction": "What is the atomic number of helium?",
    "output": "The atomic number of helium is 2.",
},
```

- 이 코드는 GPU가 필요하지 않으며 노트북에서 실행됩니다 (M3 MacBook Air에서 테스트됨)

*여기서 생성된 지시 데이터셋은 교육 목적입니다. 그러나 사용자는 Meta AI의 Llama 3와 관련된 라이선스 계약 조건을 준수해야 할 의무가 있습니다.*

In [1]:
from importlib.metadata import version

pkgs = [
    "tqdm",    # 진행률 표시줄
]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## Ollama 설치 및 Llama 3 다운로드

- Ollama는 LLM을 효율적으로 실행하는 애플리케이션입니다
- 이는 효율성을 극대화하기 위해 순수 C/C++로 LLM을 구현한 [llama.cpp](https://github.com/ggerganov/llama.cpp)를 감싸는 래퍼입니다
- 이것은 텍스트 생성(추론)을 위한 LLM 사용 도구이며, LLM 학습이나 미세조정을 위한 것이 아님에 주목하세요
- 아래 코드를 실행하기 전에 [https://ollama.com](https://ollama.com)을 방문하여 지침을 따라 ollama를 설치하세요 (예: "Download" 버튼을 클릭하고 운영 체제용 ollama 애플리케이션을 다운로드)

- macOS 및 Windows 사용자의 경우 다운로드한 ollama 애플리케이션을 클릭하세요. 명령줄 사용을 설치하라는 메시지가 표시되면 "yes"라고 답하세요
- Linux 사용자는 ollama 웹사이트에 제공된 설치 명령어를 사용할 수 있습니다

- 일반적으로 명령줄에서 ollama를 사용하기 전에 ollama 애플리케이션을 시작하거나 별도의 터미널에서 `ollama serve`를 실행해야 합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- ollama 애플리케이션이나 `ollama serve`가 실행 중인 상태에서 다른 터미널의 명령줄에서 다음 명령을 실행하여 80억 파라미터의 Llama 3 모델을 시도해보세요 (이 명령을 처음 실행할 때 4.7GB의 저장 공간을 차지하는 모델이 자동으로 다운로드됩니다)

```bash
# 8B 모델
ollama run llama3
```


출력은 다음과 같습니다:

```
$ ollama run llama3
pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
removing any unused layers 
success 
```

- `llama3`는 지시 미세조정된 80억 파라미터 Llama 3 모델을 가리킵니다

- 또는 머신이 지원하는 경우 `llama3`를 `llama3:70b`로 바꾸어 더 큰 700억 파라미터 Llama 3 모델을 사용할 수도 있습니다

- 다운로드가 완료된 후 모델과 채팅할 수 있는 명령줄 프롬프트가 나타납니다

- "What do llamas eat?"와 같은 프롬프트를 시도해보세요. 다음과 비슷한 출력을 반환해야 합니다:

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- `/bye` 입력을 사용하여 이 세션을 종료할 수 있습니다

## Ollama의 REST API 사용

- 이제 모델과 상호작용하는 대안적인 방법은 다음 함수를 통해 Python의 REST API를 사용하는 것입니다
- 이 노트북의 다음 셀들을 실행하기 전에 위에서 설명한 대로 ollama가 여전히 실행 중인지 확인하세요:
  - 터미널에서 `ollama serve`
  - ollama 애플리케이션
- 다음으로 모델을 쿼리하기 위해 다음 코드 셀을 실행하세요

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [2]:
import urllib.request
import json

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat", role="user"):
    # 데이터 페이로드를 딕셔너리로 생성
    data = {
        "model": model,
        "seed": 123,        # 결정론적 응답을 위함
        "temperature": 1.,   # 결정론적 응답을 위함
        "top_p": 1,         
        "messages": [
            {"role": role, "content": prompt}
        ]
    }

    # 딕셔너리를 JSON 형식의 문자열로 변환하고 바이트로 인코딩
    payload = json.dumps(data).encode("utf-8")

    # 요청 객체를 생성하고 메서드를 POST로 설정하며 필요한 헤더 추가
    request = urllib.request.Request(url, data=payload, method="POST")
    request.add_header("Content-Type", "application/json")

    # 요청을 보내고 응답을 캡처
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # 응답을 읽고 디코딩
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

In [3]:
result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: They enjoy eating hay, such as alfalfa or timothy hay, which provides them with fiber, protein, and other essential nutrients.
3. Grains: Llamas may eat grains like oats, barley, or corn as a supplement to their diet.
4. Leaves: They will also munch on leaves from trees and shrubs, including clover, alfalfa, and various types of leaves.
5. Fruits and vegetables: In the wild, llamas might eat fruits and vegetables that grow in their natural habitat, such as apples, carrots, or potatoes.

In general, a llama's diet should consist of:

* 50% grasses and hay
* 20% grains (like oats or corn)
* 10% leaves and other plant material
* 5% fruits and vegetables (as treats)

It's essential to provide llamas with a balanced diet that meets their nutritional n

## 지시 추출

- 이제 논문에서 제안된 "핵"을 사용해보겠습니다: 빈 프롬프트 템플릿 `"<|begin_of_text|><|start_header_id|>user<|end_header_id|>"` 프롬프트를 제공하면, 지시 미세조정된 Llama 3 모델이 지시를 생성하게 됩니다

In [4]:
def extract_instruction(text):
    for content in text.split("\n"):
        if content:
            return content.strip()

In [5]:
query = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>"

result = query_model(query, role="assistant")
instruction = extract_instruction(result)
print(instruction)

I am trying to find a way to make my child's birthday party more special and unique. What are some creative ideas you have?


- 위에서 볼 수 있듯이, 놀랍게도 모델이 실제로 지시를 생성했습니다

## 응답 생성

- 이제 다음 단계는 해당 응답을 생성하는 것인데, 이는 단순히 지시를 입력으로 전달하여 수행할 수 있습니다

In [6]:
response = query_model(instruction, role="user")
print(response)

What an exciting question! I'd be delighted to help you come up with some creative and unique ideas to make your child's birthday party truly special!

Here are a few ideas to get you started:

1. **Themed Scavenger Hunt**: Plan a scavenger hunt based on the birthday child's favorite theme (e.g., superheroes, animals, or princesses). Hide clues and challenges throughout the party area, leading up to a final surprise.
2. **DIY Crafts Station**: Set up a craft station where kids can create their own party favors, such as customized t-shirts, crowns, or jewelry. This activity encourages creativity and makes for a memorable keepsake.
3. **Mystery Box Challenge**: Fill mystery boxes with different textures, smells, and sounds. Have the kids guess what's inside each box without looking. This game promotes problem-solving and teamwork.
4. **Indoor Camping Adventure**: Set up a cozy indoor "camping" area with sleeping bags, flashlights, and s'mores-making stations. Kids can enjoy a camping exp

## 데이터셋 생성

- 이 접근법을 임의의 수의 데이터 샘플로 확장할 수 있습니다 (길이나 품질에 대한 선택적 필터링을 적용할 수 있습니다 (예: 다른 LLM을 사용하여 생성된 데이터를 평가))
- 아래에서는 5개의 합성 지시-응답 쌍을 생성하는데, 이는 M3 MacBook Air에서 약 3분이 소요됩니다
- (지시 미세조정에 적합한 데이터셋을 생성하려면 최소 1천에서 5만 개로 늘리고 GPU에서 실행하여 더 시의적절하게 예제를 생성하는 것이 좋습니다)

**팁**

- `model="llama3"`를 `model="llama3:70b"`로 변경하면 더 높은 품질의 응답을 생성할 수 있지만, 더 많은 계산 자원이 필요합니다

In [7]:
from tqdm import tqdm

dataset_size = 5
dataset = []

for i in tqdm(range(dataset_size)):

    result = query_model(query, role="assistant")
    instruction = extract_instruction(result)
    response = query_model(instruction, role="user")
    entry = {
        "instruction": instruction,
        "output": response
    }
    dataset.append(entry)

100%|█████████████████████████████████████████████| 5/5 [02:37<00:00, 31.41s/it]


In [8]:
with open("instruction-data-llama3-7b.json", "w") as file:
    json.dump(dataset, file, indent=4)

In [9]:
!cat instruction-data-llama3-7b.json

Outputs are too large to include. Use Bash with: cat <notebook_path> | jq '.cells[25].outputs'
